In [ ]:
from mysql import connector
import cv2
import matplotlib.pyplot as plt
import os
import numpy as np


cnx = connector.connect(user='root', password='root',
                              host='172.17.0.5',
                              database='textline_app_db')
cursor = cnx.cursor()

query = ("SELECT doc_id, doc_path, output, status FROM documents WHERE created_at > '2025-04-08 17'")
cursor.execute(query)
d_id, path, polygons, status = None, None, None, None
for i, (doc_id, doc_path, output, stat)in enumerate(cursor):
    d_id, path, polygons, status = doc_id, doc_path, output, stat
    if i ==0:
        break

print(d_id)
# print(polygons)
polygons = eval(polygons)['polygons']

image = cv2.imread(os.path.join('/data3/amal.joseph/template_api/store/user_files/uploaded_documents', path))


image2 = image.copy()
for line in polygons:
    image2 = cv2.polylines(image2, [np.array(line)], color=(255, 0, 0), thickness=1, isClosed=True)

plt.imshow(image2)

In [ ]:
import imagesize
width, height = imagesize.get('/data3/amal.joseph/template_api/store/user_files/uploaded_documents/d657f788-e444-41b1-ad1b-9d30b0649106/040005.jpg')
width, height

In [1]:
from fastapi import APIRouter, Depends, HTTPException, status, Response
from sqlalchemy.orm import Session

from src.database import crud, schemas, models # Import models for type hinting
from src.api import deps
from uuid import UUID

job_id = 'f5eb1ddc-2567-421a-92d5-a67e9013c952'
job_id =UUID(job_id)
db = next(deps.get_db())
db_job = crud.get_job_by_job_id(db=db, job_id_bytes=job_id.bytes)
db_docs = crud.get_job_document_statuses(db=db, job_id_bytes=job_id.bytes)

In [5]:
from ast import literal_eval


data = []
for doc_record in db_docs:
    output = literal_eval(doc_record.output).get('polygons')

    polygons = []
    if output:
        for i, line in enumerate(output):
            new_line = {
                "id": f"line{i+1}", 
                "points": [tuple(points) for points in line]
            }
            polygons.append(new_line)

    new_doc_record = {}
    new_doc_record['image_name'] = doc_record.doc_path
    # new_doc_record['image_name'] = doc_record.doc_path.split('/')[-1]
    new_doc_record['width'], new_doc_record['height'] = doc_record.width, doc_record.height
    new_doc_record['polygons'] = polygons



    data.append(new_doc_record)
data

[{'image_name': 'f5eb1ddc-2567-421a-92d5-a67e9013c952/000001.jpg',
  'width': 1200,
  'height': 1200,
  'polygons': [{'id': 'line1',
    'points': [(66, 375),
     (67, 375),
     (68, 376),
     (69, 376),
     (70, 376),
     (71, 376),
     (72, 376),
     (73, 376),
     (74, 375),
     (75, 374),
     (76, 373),
     (77, 372),
     (78, 371),
     (79, 371),
     (80, 371),
     (81, 371),
     (82, 371),
     (83, 371),
     (84, 371),
     (85, 371),
     (86, 371),
     (87, 372),
     (88, 373),
     (89, 374),
     (90, 375),
     (91, 376),
     (92, 377),
     (93, 378),
     (94, 379),
     (95, 380),
     (96, 381),
     (97, 382),
     (98, 383),
     (99, 382),
     (100, 381),
     (101, 380),
     (102, 379),
     (103, 378),
     (104, 377),
     (105, 376),
     (106, 375),
     (107, 375),
     (108, 375),
     (109, 374),
     (110, 374),
     (111, 374),
     (112, 374),
     (113, 374),
     (114, 375),
     (115, 375),
     (116, 375),
     (117, 375),
     (1

In [6]:
for d in data:
    print(d.keys())
    print(d['image_name'], )
    for line in d['polygons']:
        print(line['id'])

dict_keys(['image_name', 'width', 'height', 'polygons'])
f5eb1ddc-2567-421a-92d5-a67e9013c952/000001.jpg
line1
line2


In [7]:
import xml.etree.ElementTree as ET
from xml.dom.minidom import parseString

def convert_to_tei(data):
    tei = ET.Element("TEI", xmlns="http://www.tei-c.org/ns/1.0")
    facsimile = ET.SubElement(tei, "facsimile")

    for page in data:
        surface = ET.SubElement(facsimile, "surface", {"xml:id": page["image_name"].split('.')[0]})
        
        ET.SubElement(
            surface, "graphic",
            {
                "url": page["image_name"],
                "width": str(page["width"]),
                "height": str(page["height"]),
            },
        )

        for poly in page["polygons"]:
            points_str = " ".join(f"{x},{y}" for x, y in poly["points"])
            ET.SubElement(
                surface, "zone",
                {
                    "xml:id": f"{page['image_name'].split('.')[0]}_{poly['id']}",
                    "points": points_str,
                    "type": "line",
                },
            )
    tree = ET.ElementTree(tei)
    tree.write('test.xml', encoding="utf-8", xml_declaration=True)
    rough_string = ET.tostring(tei, encoding="utf-8")
    reparsed = parseString(rough_string)
    pretty_xml = reparsed.toprettyxml(indent="  ")

    # Save to file with proper formatting
    with open('new_test.xml', "w", encoding="utf-8") as f:
        f.write(pretty_xml)
    return ET.tostring(tei, encoding="unicode")
xml_op = convert_to_tei(data)

In [ ]:
xml_op

In [41]:
import os
import cv2
import argparse
import numpy as np
import xml.etree.ElementTree as ET
from pdf2image import convert_from_path
from shutil import copyfile, rmtree
# TEI XML namespace
NS = {'tei': 'http://www.tei-c.org/ns/1.0'}


def process_documents(docs_dir):
    documents = [os.path.join(docs_dir, file) for file in os.listdir(docs_dir)]
    temp_dir = os.path.join(output_dir, 'tmp_folder')
    os.makedirs(temp_dir, exist_ok=True)
    for document in documents:
        if document.endswith('.pdf'):
            doc_images = convert_from_path(document, use_cropbox=True)
            for i, image in enumerate(doc_images):
                temp_image_name = document.split('/')[-1].replace('.pdf', '') + f'_page_{i+1}.jpg'
                temp_image_name = os.path.join(temp_dir, temp_image_name)
                image.save(temp_image_name)
        else:
            copyfile(document, os.path.join(temp_dir, document.split('/')[-1]))
    return temp_dir
    
def run(docs_dir, xml_path, output_dir):
    temp_dir = process_documents(docs_dir)
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    facsimile = root.find('tei:facsimile', NS)
    if facsimile is None:
        print("No <facsimile> tag found.")
        return
    
    for surface in facsimile.findall('tei:surface', NS):
        surface_id = surface.get('{http://www.w3.org/XML/1998/namespace}id')
        graphic = surface.find('tei:graphic', NS)
        if graphic is None:
            continue
        image_filename = graphic.get('url')
        image_path = os.path.join(temp_dir, image_filename)
        if os.path.exists(image_path):
            image = cv2.imread(image_path)
            # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            output_path = os.path.join(output_dir, image_filename)
            for zone in surface.findall('tei:zone', NS):
                points_str = zone.get('points')
                if not points_str:
                    continue
                points = [tuple(map(int, pt.split(','))) for pt in points_str.split()]
                pts = np.array(points)
                image = cv2.polylines(image, [pts], isClosed=True, color=(0, 255, 0), thickness=2)
            cv2.imwrite(output_path, image)
        else:
            print('Skipping {image_filename}. ')
    
    rmtree(temp_dir)
    


            

    

if __name__ == "__main__":
    image_root = '/data3/amal.joseph/temp_dir/sample_docs'
    tei = '/data3/amal.joseph/temp_dir/92BC03E8-1F33-414A-BFF5-46F67C217243.xml'
    output_dir = '/data3/amal.joseph/temp_dir/extracted_polygons'
    
    run(image_root, tei, output_dir)
    
    
# /data3/amal.joseph/temp_dir/extracted_polygons/tmp_folder/data_jain_mscripts_Chandibazaar_chandibajar_0011.2_images_0358.jpg

In [ ]:
import cv2
import numpy as np

def crop_image(image, coordinates, output_path):
    points = np.array(coordinates, dtype=np.int32)
    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [points], (255))
    
    # Bitwise AND the mask with the original image to isolate the desired area
    cropped_image = cv2.bitwise_and(image, image, mask=mask)
    x, y, w, h = cv2.boundingRect(points)
    cropped_image = cropped_image[y:y+h, x:x+w]

    cv2.imwrite(output_path, cropped_image)


image_path = '/data3/amal.joseph/temp_dir/sample_docs/000002.jpg'
coordinates = points
output_path = '/data3/amal.joseph/temp_dir/crop.jpg'
crop_image(image_path, coordinates, output_path)

Cropped image saved to /data3/amal.joseph/temp_dir/crop.jpg


In [45]:
points="790,313 791,312 792,311 793,310 794,311 795,312 796,313 797,314 798,315 799,314 800,313 801,313 802,312 803,312 804,311 805,310 806,309 807,310 808,311 809,312 810,313 811,314 812,314 813,313 814,313 815,312 816,311 817,311 818,311 819,312 820,313 821,313 822,314 823,313 824,312 825,313 826,313 827,314 828,315 829,315 830,315 831,315 832,315 833,316 834,316 835,315 836,315 837,315 838,315 839,315 840,314 841,314 842,314 843,314 844,315 845,315 846,315 847,314 848,314 849,314 850,315 851,315 852,315 853,316 854,317 855,317 856,316 857,315 858,315 859,314 860,314 861,314 862,315 863,315 864,315 865,316 866,316 867,316 868,315 869,314 870,313 871,314 872,315 873,315 874,315 875,314 876,313 877,312 878,312 879,312 880,312 881,312 882,312 883,311 884,311 885,311 886,312 887,313 888,314 889,313 890,312 891,311 892,310 893,309 894,308 895,307 896,306 897,305 898,304 899,305 900,306 901,307 902,308 903,307 904,308 905,309 906,309 907,309 908,308 909,308 910,308 911,309 912,310 913,311 914,312 915,313 916,314 917,313 918,313 919,313 920,313 921,314 922,314 923,313 924,312 925,313 926,312 927,311 928,310 929,309 930,308 931,307 932,306 933,305 934,304 935,303 936,302 937,301 938,300 939,299 940,298 941,297 942,296 943,295 944,295 945,295 946,295 947,294 948,295 949,294 950,294 951,295 952,295 953,295 954,295 955,295 956,295 957,295 958,295 959,295 960,294 961,294 962,294 963,294 964,294 965,295 966,295 967,295 968,295 969,295 970,295 971,295 972,295 973,295 974,295 975,295 976,295 977,295 978,295 979,294 980,294 981,293 982,293 983,294 984,295 985,295 986,295 987,295 988,295 989,294 990,294 991,293 992,294 993,295 994,295 995,295 996,295 997,295 998,295 999,295 1000,295 1001,295 1002,295 1003,295 1004,295 1005,295 1006,295 1007,294 1008,293 1009,292 1010,292 1011,292 1012,293 1013,294 1014,295 1015,294 1016,295 1017,294 1018,295 1019,295 1020,295 1021,295 1022,295 1023,295 1024,295 1025,295 1026,295 1027,295 1028,295 1029,295 1030,295 1031,295 1032,295 1033,294 1034,294 1035,295 1036,294 1037,294 1038,294 1039,294 1040,294 1041,295 1042,295 1043,294 1044,294 1045,295 1046,294 1047,294 1048,294 1049,295 1050,295 1051,294 1052,293 1053,293 1054,292 1055,293 1056,293 1057,293 1058,293 1059,293 1060,293 1061,293 1062,293 1063,293 1064,293 1065,293 1066,293 1067,293 1068,293 1069,293 1070,293 1071,293 1072,293 1073,293 1074,293 1075,293 1076,292 1077,292 1078,292 1079,291 1080,291 1081,291 1082,290 1083,290 1084,290 1085,291 1086,291 1087,291 1088,291 1089,291 1090,291 1091,290 1092,290 1093,291 1094,290 1095,290 1096,290 1097,290 1098,290 1099,290 1100,290 1101,290 1102,290 1103,290 1104,289 1105,289 1106,289 1107,289 1108,289 1109,289 1110,289 1111,289 1112,289 1113,289 1114,289 1115,289 1116,289 1117,290 1118,290 1119,290 1120,289 1121,290 1122,290 1123,289 1124,289 1125,289 1126,289 1127,288 1128,287 1129,286 1130,286 1131,287 1132,288 1133,289 1134,290 1135,291 1136,292 1137,293 1138,294 1139,295 1140,295 1141,295 1142,295 1143,295 1144,295 1145,295 1146,295 1147,295 1148,295 1149,295 1150,295 1151,295 1152,295 1153,295 1154,295 1155,295 1156,295 1157,295 1158,295 1159,295 1160,295 1161,295 1162,295 1163,295 1164,295 1165,295 1166,295 1167,295 1168,295 1169,295 1170,295 1171,295 1172,294 1173,295 1174,294 1175,294 1176,294 1177,294 1178,293 1179,293 1180,294 1181,295 1182,295 1183,295 1184,295 1185,295 1186,295 1187,295 1188,295 1189,294 1190,293 1191,292 1192,292 1193,293 1194,294 1195,295 1196,295 1197,294 1198,294 1199,294 1200,293 1201,292 1202,291 1203,290 1204,291 1205,292 1206,292 1207,293 1208,294 1209,295 1210,295 1211,294 1212,293 1213,294 1214,294 1215,295 1216,294 1217,294 1218,294 1219,293 1220,292 1221,293 1222,293 1223,294 1224,295 1225,295 1226,294 1227,294 1228,293 1229,293 1230,293 1231,293 1232,293 1233,293 1234,294 1235,294 1236,294 1237,295 1238,295 1239,295 1240,295 1241,294 1242,294 1243,294 1244,295 1245,295 1246,295 1247,295 1248,295 1249,294 1250,293 1251,293 1252,293 1253,293 1254,293 1255,293 1256,293 1257,292 1258,292 1259,293 1260,293 1261,293 1262,293 1263,292 1264,291 1265,291 1266,290 1267,289 1268,288 1269,287 1270,286 1271,287 1272,286 1273,286 1274,287 1275,287 1276,286 1277,286 1278,287 1279,288 1280,289 1281,289 1282,288 1283,289 1284,290 1285,289 1286,290 1287,290 1288,291 1289,290 1290,290 1291,290 1292,289 1293,288 1294,288 1295,288 1296,288 1297,288 1298,287 1299,286 1300,286 1301,285 1302,286 1303,286 1304,287 1305,286 1306,287 1307,287 1308,286 1309,286 1310,285 1311,284 1312,285 1313,285 1314,285 1315,284 1316,285 1317,285 1318,286 1319,286 1320,286 1321,287 1322,288 1323,288 1324,287 1325,286 1326,286 1327,285 1328,284 1329,285 1330,285 1331,285 1332,285 1333,284 1334,284 1335,284 1336,284 1337,284 1338,284 1339,285 1340,286 1341,286 1342,286 1343,286 1344,287 1345,286 1346,285 1347,284 1348,283 1349,284 1350,285 1351,286 1352,287 1353,287 1354,287 1355,287 1356,286 1357,286 1358,286 1359,285 1360,285 1361,285 1362,285 1363,285 1364,285 1365,284 1366,283 1367,283 1368,283 1369,284 1370,283 1371,283 1372,283 1373,282 1374,282 1375,281 1376,281 1377,282 1378,281 1379,281 1380,281 1381,282 1382,282 1383,283 1384,283 1385,283 1386,284 1387,283 1388,283 1389,284 1390,284 1391,284 1392,284 1393,284 1394,284 1395,284 1396,284 1397,284 1398,284 1399,283 1400,284 1401,284 1402,283 1403,282 1404,282 1405,282 1406,283 1407,283 1408,283 1409,284 1410,284 1411,283 1412,282 1413,281 1414,282 1415,283 1416,284 1417,283 1418,283 1419,282 1420,283 1421,284 1422,283 1423,284 1424,285 1425,286 1426,286 1427,285 1428,284 1429,283 1430,282 1431,283 1432,283 1433,284 1434,283 1435,284 1436,284 1437,285 1438,284 1439,285 1440,285 1441,285 1442,285 1443,285 1444,285 1445,285 1446,285 1447,284 1448,285 1449,286 1450,287 1451,286 1452,285 1453,284 1454,284 1455,284 1456,284 1457,285 1458,285 1459,285 1460,285 1461,285 1462,285 1463,285 1464,285 1465,285 1466,285 1467,285 1468,285 1469,285 1470,285 1471,284 1472,284 1473,283 1474,284 1475,284 1476,284 1477,284 1478,285 1479,285 1480,285 1481,285 1482,285 1483,286 1484,285 1485,284 1486,283 1487,283 1488,283 1489,283 1490,283 1491,283 1492,283 1493,284 1494,284 1495,284 1496,285 1497,285 1498,285 1499,284 1500,283 1501,283 1502,283 1503,282 1504,282 1505,282 1506,283 1507,284 1508,284 1509,285 1510,284 1511,283 1512,282 1513,281 1514,280 1515,279 1516,278 1517,277 1518,276 1519,276 1520,277 1521,277 1522,276 1523,276 1524,276 1525,276 1526,277 1527,276 1528,275 1529,275 1530,276 1531,275 1532,274 1533,273 1534,274 1535,274 1536,274 1537,274 1538,273 1539,273 1540,272 1541,272 1542,273 1543,272 1544,271 1545,272 1546,273 1547,273 1548,272 1549,271 1550,272 1551,273 1552,274 1553,274 1554,273 1555,273 1556,273 1557,272 1558,273 1559,272 1560,271 1561,272 1562,272 1563,273 1564,274 1565,275 1566,276 1567,277 1568,278 1569,278 1570,277 1571,276 1572,277 1573,277 1574,278 1575,278 1576,278 1577,277 1578,276 1579,276 1580,276 1581,277 1582,277 1583,277 1584,277 1585,278 1586,279 1587,280 1588,281 1589,280 1590,279 1591,278 1592,278 1593,277 1594,276 1595,275 1596,274 1597,273 1598,272 1599,271 1600,270 1601,269 1602,268 1603,268 1604,267 1605,268 1606,269 1607,270 1608,269 1609,270 1610,270 1611,270 1612,271 1613,271 1614,272 1615,272 1616,273 1617,273 1618,273 1619,274 1620,273 1621,274 1622,275 1623,276 1624,277 1625,278 1626,279 1627,280 1628,281 1629,282 1630,283 1631,284 1632,284 1633,283 1634,284 1635,283 1636,283 1637,283 1638,283 1639,283 1640,283 1641,284 1642,285 1643,284 1644,284 1645,284 1646,284 1647,284 1648,285 1649,284 1650,284 1651,284 1652,283 1653,282 1654,281 1655,282 1656,282 1657,282 1658,281 1659,281 1660,281 1661,281 1662,281 1663,280 1664,281 1665,281 1666,280 1667,279 1668,278 1669,277 1670,276 1671,275 1672,274 1673,273 1674,272 1675,271 1676,270 1677,270 1678,271 1679,272 1680,271 1681,271 1682,271 1683,270 1684,271 1685,270 1686,270 1687,270 1688,269 1689,270 1690,271 1691,271 1692,270 1693,271 1694,271 1695,271 1696,271 1697,271 1698,271 1699,272 1700,272 1701,272 1702,272 1703,272 1704,273 1705,273 1706,274 1707,274 1708,275 1709,275 1710,275 1711,274 1712,273 1713,274 1714,274 1715,275 1716,276 1717,276 1718,277 1719,277 1720,278 1721,279 1722,280 1723,280 1724,280 1725,280 1726,281 1727,281 1728,281 1729,281 1730,281 1731,281 1732,281 1733,280 1734,280 1735,280 1736,280 1737,280 1738,280 1739,280 1740,281 1741,281 1742,281 1743,281 1744,281 1745,281 1746,280 1747,280 1748,279 1749,278 1750,277 1751,276 1752,275 1753,274 1754,273 1755,272 1756,271 1757,270 1758,269 1759,268 1760,267 1761,266 1762,266 1763,266 1764,265 1765,265 1766,265 1767,264 1768,264 1769,264 1770,264 1771,264 1772,264 1773,265 1774,265 1775,266 1776,266 1777,266 1778,266 1779,266 1780,266 1781,267 1782,266 1783,267 1784,267 1785,266 1786,266 1787,266 1788,266 1789,265 1790,266 1791,267 1792,267 1793,267 1794,267 1795,267 1796,267 1797,268 1798,269 1799,268 1800,268 1801,267 1802,266 1803,266 1804,266 1805,266 1806,266 1807,267 1808,267 1809,268 1810,269 1811,270 1812,271 1813,272 1814,273 1815,274 1816,274 1817,274 1818,274 1819,274 1820,274 1821,274 1822,275 1823,275 1824,275 1825,274 1826,273 1827,272 1828,273 1829,274 1830,274 1831,274 1832,274 1833,273 1834,274 1835,275 1836,275 1837,274 1838,273 1839,272 1840,272 1841,271 1842,270 1843,269 1844,269 1845,269 1846,270 1847,271 1848,271 1849,270 1850,269 1851,268 1852,267 1853,266 1854,265 1855,265 1856,266 1857,266 1858,266 1859,267 1860,268 1861,269 1862,270 1863,271 1864,272 1865,273 1866,274 1867,275 1868,276 1869,277 1870,278 1871,279 1872,280 1873,281 1874,282 1875,283 1876,283 1877,282 1878,281 1879,281 1880,281 1881,282 1882,282 1883,281 1884,281 1885,280 1886,280 1887,281 1888,282 1889,282 1890,281 1891,282 1892,281 1893,280 1894,279 1895,278 1896,277 1897,276 1898,275 1899,275 1900,274 1901,274 1902,275 1903,276 1904,277 1905,277 1906,276 1907,275 1908,274 1909,273 1910,272 1911,271 1912,270 1913,270 1914,269 1915,269 1916,270 1917,271 1918,272 1919,273 1920,273 1921,272 1922,271 1923,270 1924,269 1925,269 1926,268 1927,269 1928,270 1929,271 1930,270 1931,269 1932,269 1933,269 1934,270 1935,270 1936,269 1937,270 1938,269 1939,269 1940,269 1941,268 1942,267 1943,267 1944,267 1945,268 1946,268 1947,268 1948,268 1949,268 1950,267 1951,268 1952,267 1953,268 1954,267 1955,268 1956,269 1957,269 1958,269 1959,269 1960,268 1961,268 1962,268 1963,267 1964,266 1965,265 1966,264 1967,263 1968,262 1969,261 1970,260 1971,259 1972,258 1973,257 1974,256 1975,255 1976,254 1977,253 1978,253 1979,253 1980,253 1981,253 1982,253 1983,253 1984,253 1985,253 1986,253 1987,253 1988,253 1989,253 1990,253 1991,253 1992,253 1993,253 1994,253 1995,253 1996,253 1997,253 1998,253 1999,253 2000,253 2001,253 2002,253 2003,253 2004,253 2005,253 2006,253 2007,253 2008,253 2009,253 2010,253 2011,253 2012,253 2013,253 2014,253 2015,252 2016,251 2017,251 2018,252 2019,252 2020,252 2021,251 2022,250 2023,249 2024,249 2025,250 2026,250 2027,251 2028,252 2029,253 2030,253 2031,252 2032,251 2033,250 2034,249 2035,248 2036,247 2037,247 2038,248 2039,247 2040,246 2041,245 2042,244 2043,243 2044,242 2045,241 2046,240 2047,239 2048,240 2049,240 2050,240 2051,240 2052,241 2053,241 2054,241 2055,241 2056,242 2057,241 2058,241 2059,241 2060,241 2061,242 2062,241 2063,240 2064,240 2065,241 2066,242 2067,243 2068,244 2069,245 2070,246 2071,247 2072,248 2073,249 2074,248 2075,248 2076,247 2077,248 2078,249 2079,250 2080,251 2081,251 2082,251 2083,252 2084,253 2085,253 2086,253 2087,253 2088,253 2089,253 2090,253 2091,252 2092,253 2093,253 2094,253 2095,253 2096,253 2097,253 2098,253 2099,253 2100,253 2101,253 2102,253 2103,253 2104,253 2105,253 2106,252 2107,251 2108,252 2109,252 2110,253 2111,253 2112,253 2113,252 2114,251 2115,251 2116,252 2117,252 2118,253 2119,253 2120,252 2121,251 2122,251 2123,251 2124,252 2125,251 2126,251 2127,251 2128,250 2129,249 2130,249 2131,248 2132,249 2133,249 2134,248 2135,247 2136,248 2137,249 2138,250 2139,250 2140,250 2141,250 2142,250 2143,251 2144,252 2145,253 2146,253 2147,253 2148,253 2149,253 2150,253 2151,253 2152,253 2153,253 2154,253 2155,253 2156,253 2157,253 2158,253 2159,253 2160,253 2161,252 2162,251 2163,252 2164,251 2165,250 2166,249 2167,249 2168,249 2169,250 2170,251 2171,251 2172,251 2173,251 2174,252 2175,253 2176,252 2177,252 2178,251 2179,250 2180,249 2181,248 2182,247 2183,246 2184,245 2185,244 2186,243 2187,242 2188,241 2189,242 2190,243 2191,244 2192,245 2193,245 2194,244 2195,243 2196,242 2197,241 2198,241 2199,242 2200,241 2201,241 2202,240 2203,241 2204,242 2205,243 2206,244 2207,244 2208,244 2209,243 2210,242 2211,241 2212,240 2213,241 2214,241 2215,240 2216,240 2217,240 2218,239 2219,240 2220,240 2221,241 2222,242 2223,242 2224,242 2225,241 2226,242 2227,243 2228,244 2229,245 2230,246 2231,247 2232,248 2233,247 2234,246 2235,245 2236,245 2237,246 2238,247 2239,246 2240,246 2241,246 2242,245 2243,245 2244,245 2245,246 2246,247 2247,246 2248,245 2249,244 2250,243 2251,243 2252,243 2253,244 2254,244 2255,245 2256,246 2257,246 2258,245 2259,244 2260,243 2261,242 2262,241 2263,240 2264,239 2265,240 2266,241 2267,242 2268,242 2269,242 2270,242 2271,242 2272,242 2273,241 2274,241 2275,240 2276,241 2277,242 2278,242 2279,242 2280,241 2281,240 2282,239 2283,238 2284,238 2285,238 2286,239 2287,240 2288,241 2289,240 2290,239 2291,238 2292,237 2293,238 2294,239 2295,240 2296,241 2297,242 2298,242 2299,242 2300,241 2301,240 2302,239 2303,238 2304,237 2305,236 2306,235 2307,234 2308,235 2309,235 2310,235 2311,235 2312,235 2313,235 2314,236 2315,236 2316,236 2317,236 2318,237 2319,238 2320,238 2321,237 2322,237 2323,237 2324,236 2325,237 2326,238 2327,239 2328,239 2329,239 2330,239 2331,238 2332,237 2333,237 2334,237 2335,238 2336,238 2337,237 2338,237 2339,237 2340,236 2341,237 2342,237 2343,238 2344,237 2345,238 2346,239 2347,239 2348,240 2349,240 2350,239 2351,238 2352,238 2353,238 2354,239 2355,239 2356,239 2357,239 2358,238 2359,237 2360,236 2361,235 2362,234 2363,233 2364,232 2365,233 2366,234 2367,235 2368,236 2369,237 2370,238 2371,238 2372,238 2373,237 2374,238 2375,237 2376,238 2377,238 2378,238 2379,237 2380,237 2381,238 2382,238 2383,239 2384,239 2385,238 2386,237 2387,238 2388,238 2389,238 2390,238 2391,239 2392,239 2393,239 2394,238 2395,237 2396,237 2397,237 2398,236 2399,235 2400,236 2401,236 2402,236 2403,235 2404,234 2405,234 2406,234 2407,234 2408,235 2409,235 2410,234 2411,234 2412,234 2413,234 2414,234 2415,235 2416,234 2417,235 2418,236 2419,237 2420,237 2421,236 2422,235 2423,234 2424,235 2425,235 2426,235 2427,236 2428,237 2429,237 2430,237 2431,236 2432,237 2433,238 2434,238 2435,238 2436,237 2437,237 2438,236 2439,235 2440,234 2441,233 2442,232 2443,231 2444,230 2445,229 2446,228 2447,227 2448,226 2449,225 2450,224 2451,224 2452,223 2453,222 2454,221 2455,220 2456,219 2457,218 2458,217 2459,216 2460,215 2461,214 2462,214 2463,214 2464,214 2465,214 2466,214 2467,214 2468,214 2469,214 2470,214 2471,214 2472,214 2473,214 2474,214 2475,214 2476,214 2477,214 2478,214 2479,214 2480,214 2481,214 2482,214 2483,214 2484,214 2485,214 2486,214 2487,214 2488,214 2489,214 2490,214 2491,214 2492,214 2493,214 2494,214 2495,214 2496,214 2497,214 2498,214 2499,214 2500,214 2501,214 2502,214 2503,214 2504,214 2505,214 2506,214 2507,214 2508,214 2509,214 2510,214 2511,214 2512,214 2513,214 2514,214 2515,214 2516,214 2517,214 2518,214 2519,214 2520,214 2521,214 2522,214 2523,214 2524,214 2525,214 2526,214 2527,214 2528,214 2529,214 2530,214 2531,214 2532,214 2533,214 2534,214 2535,214 2536,214 2537,214 2538,214 2539,214 2540,214 2541,214 2542,214 2543,214 2544,214 2545,214 2546,214 2547,214 2548,214 2549,214 2550,214 2551,214 2552,214 2553,214 2554,214 2555,214 2556,214 2557,214 2558,214 2559,214 2560,214 2561,214 2562,214 2563,214 2564,214 2565,214 2566,214 2567,214 2568,214 2569,214 2570,214 2571,214 2572,214 2573,214 2574,214 2575,214 2576,214 2577,214 2578,214 2579,214 2580,214 2581,214 2582,214 2583,214 2584,214 2585,214 2586,214 2587,214 2588,214 2589,214 2590,214 2591,214 2592,214 2593,214 2594,214 2595,214 2596,214 2597,214 2598,214 2599,214 2600,214 2601,213 2602,212 2603,213 2604,212 2605,213 2606,214 2607,214 2608,215 2609,215 2610,214 2611,214 2612,214 2613,214 2614,214 2615,214 2616,214 2617,214 2618,214 2619,214 2620,215 2621,215 2622,215 2623,215 2624,215 2625,215 2626,215 2627,215 2628,215 2629,215 2630,215 2631,215 2632,215 2633,216 2634,215 2635,215 2636,215 2637,215 2638,214 2639,214 2640,214 2641,214 2642,215 2643,215 2644,215 2645,215 2646,215 2647,215 2648,215 2649,215 2650,215 2651,216 2652,216 2653,216 2654,217 2655,217 2656,217 2657,217 2658,216 2659,217 2660,217 2661,216 2662,216 2663,215 2664,215 2665,215 2666,215 2667,215 2668,215 2669,215 2670,215 2671,215 2672,215 2673,214 2674,213 2675,212 2676,213 2677,214 2678,215 2679,216 2680,216 2681,215 2682,215 2683,215 2684,215 2685,215 2686,214 2687,213 2688,212 2689,211 2690,211 2691,211 2692,211 2693,211 2694,211 2695,211 2696,212 2697,212 2698,212 2699,213 2700,214 2701,214 2702,214 2703,214 2704,214 2705,215 2706,215 2707,215 2708,215 2709,215 2710,215 2711,214 2712,214 2713,214 2714,214 2715,214 2716,214 2717,214 2718,214 2719,214 2720,214 2721,215 2722,215 2723,215 2724,215 2725,215 2726,215 2727,215 2728,215 2729,215 2730,215 2731,215 2732,215 2733,215 2734,214 2735,213 2736,212 2737,212 2738,211 2739,211 2740,211 2741,212 2742,212 2743,212 2744,212 2745,212 2746,212 2747,212 2748,212 2749,212 2750,213 2751,214 2752,215 2753,215 2754,215 2755,215 2756,215 2757,215 2758,215 2759,214 2760,213 2761,212 2762,211 2763,211 2764,211 2765,211 2766,212 2767,213 2768,214 2769,215 2770,215 2771,215 2772,215 2773,215 2774,215 2775,215 2776,215 2777,215 2778,214 2779,213 2780,212 2781,212 2782,213 2783,214 2784,215 2785,215 2786,215 2787,215 2788,215 2789,215 2790,214 2791,213 2792,212 2793,211 2794,211 2795,211 2796,211 2797,212 2798,213 2799,214 2800,213 2801,212 2802,212 2803,212 2804,212 2805,212 2806,212 2807,212 2808,212 2809,212 2810,212 2811,213 2812,214 2813,215 2814,214 2815,213 2816,212 2817,212 2818,213 2819,213 2820,212 2821,213 2822,214 2823,213 2824,212 2825,211 2826,212 2827,213 2828,214 2829,215 2830,215 2831,215 2832,215 2833,215 2834,214 2835,213 2836,212 2837,213 2838,214 2839,215 2840,215 2841,215 2842,214 2843,213 2844,212 2845,211 2846,211 2847,211 2848,211 2849,211 2850,212 2851,212 2852,212 2853,212 2854,212 2855,212 2856,212 2857,212 2858,211 2859,212 2860,212 2861,211 2862,212 2863,213 2864,212 2865,212 2866,212 2867,212 2868,212 2869,212 2870,212 2871,211 2872,212 2873,212 2874,213 2875,212 2876,213 2877,213 2878,212 2879,211 2880,212 2881,213 2882,214 2883,215 2884,214 2885,213 2886,212 2887,211 2888,212 2889,213 2890,214 2891,214 2892,214 2893,214 2894,214 2895,214 2896,215 2897,215 2898,215 2899,215 2900,215 2901,215 2902,215 2903,215 2904,215 2905,215 2906,215 2907,215 2908,215 2909,215 2910,215 2911,215 2912,215 2913,215 2914,215 2915,215 2916,215 2917,215 2918,215 2919,215 2920,215 2921,215 2922,215 2923,215 2924,215 2925,215 2926,215 2927,215 2928,215 2929,215 2930,215 2931,215 2932,215 2933,215 2934,214 2935,213 2936,212 2937,212 2938,212 2939,212 2940,212 2941,212 2942,211 2943,211 2944,211 2945,211 2946,211 2947,211 2948,211 2949,212 2950,213 2951,213 2952,212 2953,212 2954,213 2955,214 2956,215 2957,215 2958,215 2959,215 2960,215 2961,214 2962,213 2963,212 2964,212 2965,211 2966,212 2967,213 2968,214 2969,214 2970,214 2971,214 2972,214 2973,214 2974,214 2975,213 2976,212 2977,211 2978,212 2979,212 2980,213 2981,214 2982,215 2983,215 2984,215 2985,215 2986,215 2987,215 2988,215 2989,214 2990,214 2991,214 2992,214 2993,214 2994,214 2995,214 2996,214 2997,214 2998,214 2999,214 3000,214 3001,214 3002,214 3003,214 3004,215 3005,215 3006,215 3007,215 3008,215 3009,215 3010,215 3011,216 3012,217 3013,216 3014,216 3015,215 3016,215 3017,215 3018,215 3019,215 3020,215 3021,215 3022,215 3023,215 3024,215 3025,216 3026,217 3027,217 3028,217 3029,217 3030,217 3031,217 3032,217 3033,217 3034,218 3035,218 3036,218 3037,218 3038,218 3039,217 3040,217 3041,216 3042,216 3043,216 3044,216 3045,217 3046,217 3047,217 3048,217 3049,217 3050,217 3051,217 3052,217 3053,217 3054,217 3055,217 3056,217 3057,217 3058,217 3059,216 3060,215 3061,214 3062,214 3063,213 3064,213 3065,214 3066,214 3067,214 3068,214 3069,213 3070,214 3071,214 3072,215 3073,214 3074,214 3075,214 3076,213 3077,213 3078,214 3079,215 3080,216 3081,215 3082,214 3083,213 3084,213 3085,213 3086,213 3087,213 3088,213 3089,213 3090,213 3091,213 3092,213 3093,213 3094,214 3095,215 3096,216 3097,217 3098,217 3099,216 3100,215 3101,214 3102,213 3103,214 3104,214 3105,214 3106,215 3107,215 3108,215 3109,215 3110,215 3111,215 3112,215 3113,214 3114,214 3115,214 3116,215 3117,216 3118,217 3119,218 3120,218 3121,218 3122,218 3123,218 3124,217 3125,216 3126,215 3127,214 3128,215 3129,216 3130,217 3131,217 3132,217 3133,217 3134,217 3135,217 3136,217 3137,217 3138,217 3139,217 3140,216 3141,216 3142,216 3143,216 3144,216 3145,216 3146,216 3147,216 3148,215 3149,216 3150,217 3151,218 3152,219 3153,220 3154,220 3155,220 3156,220 3157,220 3158,220 3159,220 3160,220 3161,220 3162,220 3163,220 3164,220 3165,220 3166,220 3167,220 3168,220 3169,220 3170,220 3171,220 3172,220 3173,220 3174,220 3175,220 3176,219 3177,218 3178,217 3179,216 3180,217 3181,218 3182,219 3183,218 3184,217 3185,217 3186,218 3187,219 3188,220 3189,219 3190,218 3191,218 3192,218 3193,218 3194,218 3195,218 3196,219 3197,219 3198,218 3199,218 3200,218 3201,218 3202,217 3203,217 3204,217 3205,218 3206,219 3207,219 3208,220 3209,219 3210,218 3211,218 3212,218 3213,218 3214,218 3215,219 3216,219 3217,219 3218,219 3219,220 3220,220 3221,220 3222,220 3223,220 3224,220 3225,220 3226,220 3227,220 3228,220 3229,221 3230,221 3231,221 3232,221 3233,221 3234,221 3235,221 3236,221 3237,220 3238,220 3239,220 3240,219 3241,219 3242,219 3243,219 3244,219 3245,220 3246,220 3247,221 3248,220 3249,219 3250,219 3251,220 3252,220 3253,220 3254,221 3255,221 3256,221 3257,221 3258,221 3259,221 3260,221 3261,221 3262,222 3263,222 3264,222 3265,222 3266,222 3267,222 3268,222 3269,222 3270,222 3271,222 3272,222 3273,222 3274,222 3275,222 3276,221 3277,220 3278,221 3279,222 3280,223 3281,224 3282,224 3283,224 3284,224 3285,224 3286,224 3287,224 3288,224 3289,224 3290,224 3291,224 3292,225 3293,225 3294,224 3295,224 3296,225 3297,224 3298,224 3299,224 3300,224 3301,224 3302,224 3303,224 3304,224 3305,224 3306,225 3307,225 3308,226 3309,226 3310,226 3311,226 3312,226 3313,227 3314,227 3315,227 3316,227 3317,227 3318,226 3319,226 3320,225 3321,225 3322,225 3323,225 3324,225 3325,225 3326,225 3327,225 3328,225 3329,225 3330,225 3331,225 3332,225 3333,225 3334,225 3335,226 3336,227 3337,227 3338,227 3339,227 3340,227 3341,227 3342,226 3343,226 3344,226 3345,227 3346,227 3347,227 3348,227 3349,227 3350,227 3351,227 3352,227 3353,227 3354,227 3355,226 3356,226 3357,226 3358,226 3359,227 3360,227 3361,227 3362,227 3363,227 3364,228 3365,229 3366,230 3367,230 3368,229 3369,229 3370,228 3371,228 3372,228 3373,228 3374,228 3375,228 3376,228 3377,228 3378,228 3379,228 3380,228 3381,228 3382,228 3383,228 3384,228 3385,229 3386,229 3387,229 3388,229 3389,229 3390,228 3391,228 3392,229 3393,229 3394,230 3395,229 3396,229 3397,228 3398,228 3399,228 3400,228 3401,228 3402,228 3403,228 3404,228 3405,228 3406,228 3407,228 3408,228 3409,228 3410,229 3411,229 3412,229 3413,230 3414,230 3415,231 3416,231 3417,231 3418,230 3419,230 3420,229 3421,228 3422,228 3423,228 3424,228 3425,228 3426,229 3427,229 3428,230 3429,230 3430,230 3431,230 3432,230 3433,230 3434,230 3435,230 3436,230 3437,230 3438,231 3439,231 3440,231 3441,231 3442,231 3443,231 3444,231 3445,230 3446,230 3447,230 3448,230 3449,230 3450,230 3451,230 3452,230 3453,230 3454,230 3455,230 3456,230 3457,230 3458,230 3459,230 3460,230 3461,231 3462,231 3463,232 3464,232 3465,231 3466,232 3467,232 3468,232 3469,232 3470,232 3471,231 3472,231 3473,231 3474,231 3475,230 3476,230 3477,230 3478,230 3479,231 3480,231 3481,231 3482,231 3483,231 3484,231 3485,231 3486,231 3487,231 3488,231 3489,230 3490,230 3491,230 3492,231 3493,231 3494,230 3495,229 3496,228 3497,228 3498,228 3499,228 3500,229 3501,229 3502,230 3503,230 3504,230 3505,230 3506,230 3507,230 3508,230 3509,230 3510,231 3511,231 3512,232 3513,232 3514,232 3515,232 3516,232 3517,232 3518,232 3519,231 3520,231 3521,231 3522,231 3523,231 3524,231 3525,230 3526,230 3527,230 3528,230 3529,230 3530,230 3531,230 3532,230 3533,231 3534,231 3535,231 3536,231 3537,231 3538,232 3539,232 3540,233 3541,234 3542,235 3543,236 3544,237 3545,238 3546,239 3547,240 3548,241 3549,242 3550,241 3551,242 3552,241 3553,242 3554,243 3555,244 3556,245 3557,245 3558,244 3559,243 3560,242 3561,241 3562,240 3563,240 3564,240 3565,241 3566,242 3567,243 3568,243 3569,244 3570,244 3571,243 3572,243 3573,243 3574,243 3575,244 3576,244 3577,243 3578,243 3579,243 3580,243 3581,243 3582,243 3583,242 3584,242 3585,242 3586,243 3587,244 3588,245 3589,245 3590,245 3591,245 3592,244 3593,243 3594,243 3595,244 3596,244 3597,244 3598,243 3599,244 3600,245 3601,245 3602,245 3603,246 3604,245 3605,244 3606,243 3607,242 3608,241 3609,242 3610,242 3611,242 3612,242 3613,241 3614,240 3615,239 3616,238 3617,237 3618,236 3619,235 3620,235 3621,235 3622,235 3623,235 3624,234 3625,234 3626,234 3627,233 3628,233 3629,233 3630,233 3631,233 3632,234 3633,234 3634,234 3635,234 3636,234 3637,234 3638,234 3639,234 3640,235 3641,235 3642,236 3643,237 3644,236 3645,235 3646,235 3647,234 3648,234 3649,235 3650,235 3651,235 3652,235 3653,235 3654,235 3655,235 3656,236 3657,236 3658,236 3659,236 3660,236 3661,236 3662,236 3663,236 3664,237 3665,238 3666,239 3667,240 3668,241 3669,242 3670,243 3671,244 3672,244 3673,244 3674,245 3675,246 3676,247 3677,248 3678,249 3679,250 3680,251 3681,252 3682,253 3683,253 3684,253 3685,253 3686,253 3687,253 3688,253 3689,253 3690,253 3691,253 3692,253 3693,253 3694,253 3695,253 3696,253 3697,253 3698,253 3699,253 3700,253 3701,253 3702,253 3703,253 3704,253 3705,253 3706,253 3707,253 3708,253 3709,253 3710,253 3711,253 3712,253 3713,253 3714,253 3715,253 3716,253 3717,253 3718,253 3719,253 3720,253 3721,253 3722,253 3723,253 3724,253 3725,253 3726,253 3727,253 3728,253 3729,253 3730,253 3731,252 3732,253 3733,253 3734,253 3734,319 3733,320 3732,321 3731,322 3730,323 3729,324 3728,325 3727,326 3726,327 3725,328 3724,329 3723,330 3722,331 3721,332 3720,333 3719,334 3718,333 3717,332 3716,331 3715,330 3714,329 3713,328 3712,327 3711,326 3710,325 3709,324 3708,323 3707,322 3706,322 3705,321 3704,321 3703,322 3702,321 3701,322 3700,321 3699,321 3698,320 3697,319 3696,318 3695,317 3694,316 3693,315 3692,314 3691,313 3690,312 3689,311 3688,310 3687,309 3686,308 3685,307 3684,306 3683,305 3682,305 3681,304 3680,303 3679,303 3678,304 3677,305 3676,306 3675,307 3674,308 3673,309 3672,310 3671,311 3670,311 3669,310 3668,309 3667,310 3666,311 3665,312 3664,313 3663,314 3662,315 3661,316 3660,317 3659,318 3658,319 3657,320 3656,321 3655,322 3654,323 3653,324 3652,324 3651,323 3650,323 3649,322 3648,321 3647,321 3646,322 3645,323 3644,324 3643,325 3642,326 3641,327 3640,327 3639,328 3638,329 3637,330 3636,331 3635,332 3634,331 3633,330 3632,329 3631,330 3630,329 3629,328 3628,327 3627,328 3626,329 3625,330 3624,331 3623,332 3622,332 3621,333 3620,334 3619,333 3618,334 3617,333 3616,332 3615,331 3614,330 3613,329 3612,328 3611,328 3610,328 3609,329 3608,329 3607,330 3606,331 3605,332 3604,333 3603,332 3602,331 3601,332 3600,333 3599,334 3598,335 3597,336 3596,336 3595,336 3594,336 3593,335 3592,334 3591,333 3590,332 3589,331 3588,330 3587,330 3586,331 3585,332 3584,333 3583,334 3582,335 3581,336 3580,335 3579,334 3578,333 3577,332 3576,331 3575,330 3574,329 3573,329 3572,328 3571,328 3570,328 3569,327 3568,327 3567,327 3566,327 3565,327 3564,327 3563,326 3562,326 3561,326 3560,326 3559,326 3558,326 3557,326 3556,326 3555,325 3554,324 3553,325 3552,326 3551,327 3550,328 3549,329 3548,329 3547,329 3546,329 3545,329 3544,328 3543,327 3542,326 3541,325 3540,325 3539,326 3538,325 3537,324 3536,323 3535,322 3534,323 3533,324 3532,325 3531,325 3530,325 3529,326 3528,325 3527,326 3526,326 3525,327 3524,328 3523,328 3522,329 3521,329 3520,329 3519,329 3518,328 3517,327 3516,328 3515,329 3514,330 3513,331 3512,332 3511,331 3510,330 3509,329 3508,328 3507,327 3506,327 3505,327 3504,326 3503,327 3502,327 3501,327 3500,328 3499,328 3498,329 3497,329 3496,329 3495,330 3494,330 3493,331 3492,331 3491,330 3490,329 3489,328 3488,328 3487,328 3486,328 3485,328 3484,328 3483,328 3482,328 3481,328 3480,328 3479,328 3478,328 3477,328 3476,329 3475,328 3474,329 3473,329 3472,328 3471,329 3470,328 3469,327 3468,327 3467,327 3466,327 3465,327 3464,327 3463,327 3462,328 3461,329 3460,330 3459,329 3458,328 3457,328 3456,328 3455,327 3454,328 3453,328 3452,328 3451,328 3450,327 3449,327 3448,327 3447,327 3446,328 3445,327 3444,326 3443,325 3442,325 3441,326 3440,326 3439,326 3438,327 3437,328 3436,329 3435,330 3434,331 3433,332 3432,332 3431,331 3430,330 3429,329 3428,328 3427,327 3426,326 3425,325 3424,324 3423,323 3422,322 3421,321 3420,320 3419,319 3418,318 3417,318 3416,317 3415,316 3414,315 3413,315 3412,316 3411,317 3410,318 3409,318 3408,318 3407,318 3406,317 3405,318 3404,319 3403,319 3402,318 3401,317 3400,317 3399,317 3398,317 3397,316 3396,315 3395,315 3394,315 3393,315 3392,314 3391,314 3390,315 3389,316 3388,317 3387,318 3386,319 3385,320 3384,320 3383,321 3382,321 3381,321 3380,322 3379,322 3378,323 3377,323 3376,323 3375,323 3374,323 3373,324 3372,325 3371,326 3370,327 3369,326 3368,326 3367,326 3366,326 3365,326 3364,325 3363,325 3362,325 3361,325 3360,324 3359,323 3358,322 3357,321 3356,320 3355,319 3354,318 3353,317 3352,316 3351,315 3350,315 3349,314 3348,313 3347,312 3346,311 3345,310 3344,309 3343,308 3342,308 3341,308 3340,309 3339,310 3338,311 3337,312 3336,313 3335,314 3334,314 3333,314 3332,315 3331,315 3330,315 3329,315 3328,315 3327,314 3326,314 3325,315 3324,314 3323,314 3322,313 3321,313 3320,313 3319,314 3318,315 3317,316 3316,316 3315,316 3314,316 3313,316 3312,315 3311,314 3310,315 3309,316 3308,315 3307,314 3306,315 3305,315 3304,315 3303,316 3302,316 3301,317 3300,318 3299,319 3298,320 3297,321 3296,322 3295,323 3294,324 3293,325 3292,325 3291,326 3290,326 3289,326 3288,327 3287,328 3286,328 3285,328 3284,327 3283,327 3282,327 3281,326 3280,325 3279,324 3278,325 3277,326 3276,327 3275,328 3274,328 3273,328 3272,328 3271,328 3270,328 3269,329 3268,329 3267,329 3266,330 3265,329 3264,328 3263,329 3262,328 3261,329 3260,329 3259,329 3258,329 3257,329 3256,329 3255,329 3254,329 3253,328 3252,327 3251,326 3250,326 3249,327 3248,327 3247,326 3246,326 3245,327 3244,326 3243,325 3242,326 3241,326 3240,326 3239,326 3238,326 3237,326 3236,326 3235,326 3234,327 3233,326 3232,326 3231,326 3230,326 3229,325 3228,325 3227,324 3226,324 3225,324 3224,324 3223,323 3222,323 3221,322 3220,323 3219,324 3218,323 3217,323 3216,323 3215,324 3214,324 3213,324 3212,323 3211,323 3210,324 3209,325 3208,324 3207,323 3206,322 3205,321 3204,320 3203,319 3202,318 3201,317 3200,316 3199,315 3198,314 3197,313 3196,312 3195,311 3194,310 3193,309 3192,308 3191,307 3190,306 3189,305 3188,304 3187,304 3186,304 3185,304 3184,304 3183,304 3182,304 3181,304 3180,304 3179,304 3178,304 3177,304 3176,304 3175,304 3174,304 3173,304 3172,304 3171,305 3170,305 3169,305 3168,306 3167,307 3166,308 3165,308 3164,309 3163,309 3162,310 3161,309 3160,309 3159,309 3158,310 3157,311 3156,312 3155,313 3154,313 3153,312 3152,313 3151,312 3150,313 3149,314 3148,313 3147,313 3146,313 3145,312 3144,311 3143,310 3142,309 3141,310 3140,309 3139,309 3138,310 3137,310 3136,310 3135,311 3134,312 3133,312 3132,311 3131,312 3130,311 3129,310 3128,310 3127,311 3126,312 3125,312 3124,312 3123,311 3122,312 3121,313 3120,314 3119,313 3118,312 3117,312 3116,312 3115,313 3114,313 3113,313 3112,313 3111,312 3110,311 3109,311 3108,312 3107,311 3106,312 3105,312 3104,313 3103,312 3102,313 3101,313 3100,313 3099,313 3098,313 3097,313 3096,313 3095,313 3094,313 3093,314 3092,314 3091,314 3090,314 3089,314 3088,314 3087,314 3086,314 3085,314 3084,313 3083,314 3082,313 3081,314 3080,315 3079,314 3078,313 3077,312 3076,313 3075,312 3074,311 3073,311 3072,310 3071,311 3070,311 3069,312 3068,313 3067,312 3066,313 3065,314 3064,314 3063,315 3062,314 3061,314 3060,313 3059,313 3058,312 3057,313 3056,312 3055,312 3054,313 3053,313 3052,312 3051,312 3050,312 3049,312 3048,311 3047,310 3046,310 3045,311 3044,312 3043,312 3042,312 3041,312 3040,312 3039,311 3038,311 3037,311 3036,312 3035,311 3034,312 3033,311 3032,311 3031,311 3030,311 3029,311 3028,312 3027,312 3026,312 3025,313 3024,313 3023,313 3022,313 3021,312 3020,312 3019,312 3018,312 3017,312 3016,311 3015,310 3014,309 3013,309 3012,309 3011,310 3010,310 3009,310 3008,310 3007,311 3006,312 3005,312 3004,313 3003,312 3002,311 3001,311 3000,311 2999,311 2998,311 2997,312 2996,312 2995,311 2994,311 2993,310 2992,309 2991,309 2990,308 2989,307 2988,306 2987,307 2986,308 2985,309 2984,310 2983,311 2982,312 2981,313 2980,312 2979,312 2978,311 2977,312 2976,313 2975,314 2974,315 2973,316 2972,316 2971,315 2970,314 2969,315 2968,316 2967,317 2966,318 2965,319 2964,320 2963,319 2962,320 2961,321 2960,322 2959,323 2958,324 2957,325 2956,326 2955,327 2954,328 2953,328 2952,328 2951,328 2950,328 2949,328 2948,328 2947,328 2946,328 2945,328 2944,328 2943,328 2942,328 2941,328 2940,327 2939,327 2938,327 2937,327 2936,327 2935,327 2934,327 2933,327 2932,327 2931,327 2930,327 2929,327 2928,327 2927,327 2926,326 2925,326 2924,327 2923,327 2922,327 2921,326 2920,325 2919,324 2918,323 2917,322 2916,321 2915,320 2914,319 2913,319 2912,319 2911,320 2910,320 2909,320 2908,320 2907,320 2906,320 2905,320 2904,320 2903,321 2902,321 2901,321 2900,321 2899,321 2898,321 2897,320 2896,320 2895,320 2894,320 2893,321 2892,321 2891,321 2890,321 2889,321 2888,321 2887,320 2886,319 2885,319 2884,318 2883,319 2882,320 2881,321 2880,320 2879,320 2878,320 2877,319 2876,320 2875,320 2874,320 2873,320 2872,320 2871,319 2870,319 2869,319 2868,318 2867,318 2866,317 2865,316 2864,316 2863,316 2862,317 2861,317 2860,317 2859,318 2858,318 2857,318 2856,318 2855,318 2854,318 2853,318 2852,318 2851,318 2850,318 2849,318 2848,319 2847,319 2846,320 2845,320 2844,320 2843,320 2842,320 2841,320 2840,320 2839,319 2838,318 2837,317 2836,316 2835,315 2834,314 2833,313 2832,314 2831,315 2830,316 2829,316 2828,317 2827,318 2826,319 2825,320 2824,321 2823,322 2822,323 2821,323 2820,323 2819,323 2818,322 2817,322 2816,321 2815,321 2814,320 2813,320 2812,320 2811,320 2810,321 2809,320 2808,320 2807,320 2806,321 2805,321 2804,321 2803,322 2802,321 2801,321 2800,321 2799,321 2798,321 2797,321 2796,320 2795,319 2794,318 2793,317 2792,317 2791,317 2790,316 2789,316 2788,315 2787,315 2786,315 2785,316 2784,317 2783,318 2782,318 2781,318 2780,317 2779,318 2778,317 2777,316 2776,315 2775,315 2774,314 2773,313 2772,312 2771,311 2770,310 2769,309 2768,310 2767,311 2766,312 2765,313 2764,314 2763,315 2762,316 2761,317 2760,318 2759,318 2758,319 2757,319 2756,319 2755,318 2754,318 2753,318 2752,318 2751,318 2750,317 2749,316 2748,316 2747,317 2746,318 2745,318 2744,318 2743,319 2742,319 2741,319 2740,319 2739,320 2738,319 2737,320 2736,320 2735,321 2734,322 2733,323 2732,323 2731,323 2730,323 2729,323 2728,322 2727,322 2726,321 2725,320 2724,319 2723,318 2722,317 2721,316 2720,315 2719,314 2718,313 2717,312 2716,311 2715,310 2714,309 2713,308 2712,307 2711,306 2710,305 2709,304 2708,303 2707,302 2706,302 2705,302 2704,302 2703,302 2702,302 2701,302 2700,301 2699,301 2698,302 2697,303 2696,304 2695,305 2694,306 2693,307 2692,308 2691,309 2690,310 2689,311 2688,312 2687,313 2686,314 2685,314 2684,315 2683,316 2682,317 2681,318 2680,319 2679,320 2678,321 2677,322 2676,323 2675,323 2674,323 2673,322 2672,321 2671,321 2670,321 2669,321 2668,321 2667,321 2666,322 2665,322 2664,322 2663,321 2662,321 2661,322 2660,323 2659,323 2658,324 2657,325 2656,326 2655,327 2654,328 2653,327 2652,327 2651,326 2650,325 2649,325 2648,324 2647,323 2646,323 2645,323 2644,323 2643,323 2642,322 2641,323 2640,322 2639,321 2638,320 2637,319 2636,318 2635,319 2634,319 2633,318 2632,317 2631,316 2630,315 2629,314 2628,315 2627,315 2626,315 2625,314 2624,313 2623,312 2622,311 2621,311 2620,311 2619,312 2618,313 2617,314 2616,314 2615,314 2614,314 2613,314 2612,313 2611,314 2610,313 2609,312 2608,311 2607,310 2606,309 2605,309 2604,308 2603,308 2602,308 2601,307 2600,307 2599,307 2598,307 2597,306 2596,306 2595,307 2594,307 2593,308 2592,309 2591,310 2590,311 2589,312 2588,313 2587,314 2586,315 2585,316 2584,317 2583,318 2582,319 2581,320 2580,321 2579,322 2578,323 2577,323 2576,322 2575,322 2574,322 2573,322 2572,323 2571,323 2570,323 2569,323 2568,323 2567,323 2566,323 2565,323 2564,323 2563,322 2562,321 2561,320 2560,319 2559,318 2558,317 2557,316 2556,315 2555,315 2554,314 2553,314 2552,314 2551,314 2550,313 2549,312 2548,311 2547,310 2546,310 2545,311 2544,311 2543,311 2542,311 2541,310 2540,309 2539,308 2538,307 2537,308 2536,308 2535,308 2534,308 2533,308 2532,309 2531,308 2530,308 2529,307 2528,307 2527,308 2526,309 2525,310 2524,309 2523,308 2522,307 2521,308 2520,308 2519,308 2518,308 2517,308 2516,309 2515,310 2514,310 2513,310 2512,310 2511,311 2510,312 2509,311 2508,312 2507,312 2506,312 2505,312 2504,311 2503,310 2502,309 2501,309 2500,309 2499,309 2498,309 2497,310 2496,310 2495,310 2494,311 2493,311 2492,312 2491,312 2490,313 2489,313 2488,313 2487,313 2486,313 2485,313 2484,312 2483,311 2482,312 2481,311 2480,312 2479,312 2478,313 2477,313 2476,313 2475,314 2474,314 2473,313 2472,314 2471,314 2470,314 2469,314 2468,315 2467,316 2466,317 2465,318 2464,317 2463,317 2462,316 2461,315 2460,314 2459,313 2458,313 2457,313 2456,312 2455,313 2454,313 2453,313 2452,313 2451,313 2450,312 2449,312 2448,313 2447,313 2446,312 2445,312 2444,312 2443,312 2442,313 2441,313 2440,314 2439,314 2438,314 2437,315 2436,316 2435,317 2434,316 2433,315 2432,314 2431,314 2430,313 2429,312 2428,312 2427,312 2426,312 2425,312 2424,313 2423,314 2422,313 2421,313 2420,313 2419,313 2418,314 2417,313 2416,314 2415,314 2414,313 2413,314 2412,314 2411,314 2410,314 2409,314 2408,314 2407,315 2406,316 2405,317 2404,318 2403,318 2402,318 2401,318 2400,318 2399,319 2398,320 2397,319 2396,320 2395,320 2394,319 2393,320 2392,320 2391,320 2390,321 2389,320 2388,319 2387,318 2386,317 2385,317 2384,317 2383,316 2382,315 2381,314 2380,315 2379,316 2378,317 2377,318 2376,319 2375,320 2374,321 2373,321 2372,321 2371,322 2370,323 2369,323 2368,324 2367,325 2366,326 2365,326 2364,326 2363,327 2362,327 2361,327 2360,326 2359,325 2358,324 2357,323 2356,322 2355,321 2354,320 2353,319 2352,318 2351,317 2350,316 2349,315 2348,314 2347,313 2346,312 2345,312 2344,312 2343,312 2342,312 2341,313 2340,313 2339,313 2338,313 2337,313 2336,314 2335,313 2334,312 2333,311 2332,311 2331,311 2330,312 2329,313 2328,314 2327,313 2326,313 2325,314 2324,314 2323,315 2322,315 2321,315 2320,315 2319,315 2318,315 2317,315 2316,315 2315,315 2314,314 2313,313 2312,312 2311,313 2310,313 2309,314 2308,315 2307,315 2306,316 2305,316 2304,316 2303,317 2302,318 2301,319 2300,320 2299,321 2298,322 2297,322 2296,323 2295,323 2294,324 2293,324 2292,323 2291,323 2290,323 2289,324 2288,324 2287,324 2286,324 2285,325 2284,324 2283,325 2282,325 2281,325 2280,325 2279,326 2278,325 2277,325 2276,325 2275,325 2274,325 2273,324 2272,324 2271,325 2270,325 2269,325 2268,325 2267,325 2266,325 2265,326 2264,326 2263,326 2262,327 2261,327 2260,326 2259,325 2258,324 2257,323 2256,322 2255,321 2254,320 2253,319 2252,318 2251,317 2250,316 2249,315 2248,315 2247,315 2246,315 2245,316 2244,315 2243,316 2242,316 2241,315 2240,314 2239,314 2238,314 2237,314 2236,314 2235,314 2234,315 2233,315 2232,315 2231,316 2230,317 2229,316 2228,315 2227,316 2226,317 2225,318 2224,319 2223,320 2222,321 2221,322 2220,323 2219,324 2218,325 2217,325 2216,324 2215,324 2214,323 2213,323 2212,324 2211,324 2210,325 2209,324 2208,325 2207,325 2206,325 2205,325 2204,325 2203,326 2202,326 2201,327 2200,327 2199,327 2198,326 2197,327 2196,327 2195,327 2194,327 2193,327 2192,327 2191,327 2190,328 2189,328 2188,328 2187,328 2186,328 2185,327 2184,328 2183,329 2182,330 2181,330 2180,329 2179,328 2178,327 2177,327 2176,326 2175,326 2174,325 2173,324 2172,323 2171,322 2170,323 2169,323 2168,322 2167,321 2166,321 2165,322 2164,323 2163,324 2162,324 2161,324 2160,323 2159,324 2158,325 2157,324 2156,324 2155,325 2154,324 2153,325 2152,325 2151,326 2150,325 2149,326 2148,325 2147,324 2146,323 2145,322 2144,321 2143,322 2142,323 2141,323 2140,324 2139,325 2138,326 2137,327 2136,328 2135,328 2134,329 2133,330 2132,331 2131,331 2130,330 2129,330 2128,331 2127,331 2126,331 2125,330 2124,330 2123,331 2122,331 2121,330 2120,330 2119,329 2118,329 2117,330 2116,330 2115,330 2114,330 2113,330 2112,330 2111,330 2110,329 2109,328 2108,328 2107,328 2106,328 2105,329 2104,328 2103,327 2102,327 2101,327 2100,327 2099,327 2098,327 2097,327 2096,327 2095,326 2094,327 2093,327 2092,327 2091,328 2090,328 2089,329 2088,328 2087,327 2086,326 2085,325 2084,325 2083,325 2082,326 2081,326 2080,325 2079,324 2078,323 2077,322 2076,321 2075,320 2074,320 2073,320 2072,320 2071,319 2070,318 2069,317 2068,318 2067,318 2066,319 2065,320 2064,319 2063,318 2062,317 2061,317 2060,317 2059,317 2058,316 2057,315 2056,314 2055,313 2054,312 2053,311 2052,311 2051,311 2050,312 2049,312 2048,313 2047,312 2046,311 2045,311 2044,312 2043,313 2042,314 2041,315 2040,316 2039,316 2038,316 2037,315 2036,314 2035,313 2034,312 2033,312 2032,312 2031,313 2030,314 2029,315 2028,316 2027,317 2026,318 2025,319 2024,319 2023,318 2022,319 2021,319 2020,319 2019,318 2018,318 2017,318 2016,317 2015,318 2014,319 2013,320 2012,321 2011,322 2010,323 2009,324 2008,325 2007,324 2006,323 2005,322 2004,322 2003,322 2002,322 2001,322 2000,323 1999,324 1998,325 1997,326 1996,327 1995,328 1994,328 1993,328 1992,328 1991,329 1990,329 1989,329 1988,329 1987,328 1986,327 1985,327 1984,326 1983,325 1982,324 1981,323 1980,322 1979,321 1978,320 1977,319 1976,320 1975,321 1974,321 1973,322 1972,321 1971,320 1970,319 1969,319 1968,319 1967,319 1966,319 1965,319 1964,320 1963,320 1962,321 1961,320 1960,321 1959,321 1958,321 1957,321 1956,322 1955,322 1954,323 1953,324 1952,324 1951,325 1950,326 1949,327 1948,327 1947,327 1946,328 1945,328 1944,329 1943,330 1942,331 1941,332 1940,333 1939,333 1938,332 1937,331 1936,332 1935,332 1934,333 1933,333 1932,334 1931,334 1930,334 1929,335 1928,336 1927,337 1926,338 1925,338 1924,338 1923,338 1922,338 1921,338 1920,339 1919,340 1918,341 1917,342 1916,343 1915,344 1914,345 1913,346 1912,347 1911,347 1910,347 1909,347 1908,347 1907,347 1906,347 1905,347 1904,348 1903,347 1902,347 1901,347 1900,347 1899,346 1898,345 1897,345 1896,345 1895,345 1894,345 1893,345 1892,346 1891,345 1890,345 1889,345 1888,344 1887,345 1886,345 1885,345 1884,346 1883,346 1882,345 1881,345 1880,345 1879,345 1878,345 1877,346 1876,346 1875,346 1874,345 1873,345 1872,345 1871,345 1870,345 1869,345 1868,345 1867,345 1866,345 1865,346 1864,347 1863,348 1862,349 1861,350 1860,351 1859,352 1858,352 1857,353 1856,353 1855,352 1854,351 1853,352 1852,351 1851,351 1850,352 1849,352 1848,352 1847,351 1846,350 1845,349 1844,349 1843,350 1842,351 1841,351 1840,350 1839,350 1838,350 1837,349 1836,350 1835,351 1834,351 1833,350 1832,350 1831,351 1830,351 1829,351 1828,352 1827,353 1826,353 1825,352 1824,352 1823,352 1822,352 1821,351 1820,350 1819,349 1818,348 1817,347 1816,346 1815,345 1814,344 1813,344 1812,344 1811,344 1810,344 1809,344 1808,344 1807,344 1806,344 1805,344 1804,343 1803,343 1802,343 1801,343 1800,343 1799,343 1798,343 1797,343 1796,342 1795,342 1794,343 1793,344 1792,344 1791,345 1790,345 1789,344 1788,344 1787,344 1786,345 1785,345 1784,345 1783,345 1782,344 1781,344 1780,344 1779,343 1778,342 1777,341 1776,340 1775,339 1774,338 1773,337 1772,337 1771,336 1770,337 1769,337 1768,337 1767,338 1766,339 1765,340 1764,341 1763,342 1762,343 1761,344 1760,345 1759,346 1758,347 1757,348 1756,349 1755,348 1754,347 1753,346 1752,346 1751,346 1750,346 1749,347 1748,348 1747,347 1746,347 1745,346 1744,346 1743,346 1742,345 1741,345 1740,345 1739,345 1738,345 1737,346 1736,345 1735,345 1734,345 1733,345 1732,345 1731,345 1730,345 1729,345 1728,345 1727,345 1726,344 1725,344 1724,343 1723,343 1722,344 1721,344 1720,345 1719,346 1718,346 1717,347 1716,347 1715,347 1714,347 1713,346 1712,347 1711,346 1710,345 1709,345 1708,345 1707,345 1706,346 1705,346 1704,345 1703,345 1702,345 1701,344 1700,344 1699,344 1698,344 1697,344 1696,345 1695,346 1694,347 1693,348 1692,349 1691,350 1690,349 1689,348 1688,347 1687,347 1686,346 1685,346 1684,345 1683,344 1682,344 1681,344 1680,344 1679,345 1678,346 1677,347 1676,348 1675,349 1674,350 1673,351 1672,352 1671,352 1670,351 1669,350 1668,349 1667,350 1666,350 1665,350 1664,350 1663,349 1662,348 1661,347 1660,346 1659,345 1658,345 1657,346 1656,347 1655,347 1654,347 1653,348 1652,347 1651,346 1650,347 1649,348 1648,349 1647,350 1646,349 1645,350 1644,349 1643,350 1642,350 1641,350 1640,349 1639,349 1638,350 1637,349 1636,349 1635,348 1634,347 1633,347 1632,348 1631,348 1630,347 1629,347 1628,347 1627,348 1626,348 1625,349 1624,349 1623,350 1622,350 1621,350 1620,350 1619,350 1618,350 1617,350 1616,350 1615,350 1614,351 1613,351 1612,352 1611,353 1610,352 1609,352 1608,353 1607,352 1606,353 1605,354 1604,355 1603,356 1602,355 1601,354 1600,353 1599,352 1598,352 1597,352 1596,352 1595,353 1594,353 1593,353 1592,353 1591,352 1590,351 1589,351 1588,350 1587,349 1586,349 1585,350 1584,351 1583,352 1582,353 1581,353 1580,353 1579,353 1578,354 1577,353 1576,353 1575,354 1574,353 1573,353 1572,353 1571,353 1570,353 1569,354 1568,355 1567,355 1566,354 1565,354 1564,353 1563,352 1562,352 1561,352 1560,352 1559,352 1558,353 1557,354 1556,355 1555,354 1554,353 1553,354 1552,353 1551,354 1550,354 1549,355 1548,355 1547,354 1546,354 1545,354 1544,354 1543,354 1542,355 1541,355 1540,355 1539,355 1538,354 1537,353 1536,354 1535,354 1534,353 1533,353 1532,353 1531,354 1530,355 1529,355 1528,355 1527,355 1526,355 1525,354 1524,353 1523,353 1522,353 1521,353 1520,353 1519,353 1518,354 1517,355 1516,356 1515,357 1514,358 1513,359 1512,360 1511,361 1510,362 1509,363 1508,364 1507,365 1506,366 1505,367 1504,368 1503,369 1502,370 1501,371 1500,372 1499,373 1498,374 1497,375 1496,376 1495,377 1494,378 1493,379 1492,378 1491,377 1490,376 1489,375 1488,374 1487,373 1486,373 1485,374 1484,375 1483,375 1482,374 1481,374 1480,375 1479,376 1478,376 1477,375 1476,376 1475,376 1474,376 1473,376 1472,377 1471,378 1470,378 1469,377 1468,376 1467,375 1466,374 1465,373 1464,374 1463,373 1462,372 1461,372 1460,372 1459,373 1458,374 1457,373 1456,372 1455,371 1454,370 1453,370 1452,371 1451,371 1450,371 1449,371 1448,371 1447,371 1446,371 1445,371 1444,370 1443,370 1442,370 1441,370 1440,370 1439,370 1438,369 1437,370 1436,370 1435,369 1434,368 1433,367 1432,366 1431,365 1430,366 1429,366 1428,365 1427,365 1426,364 1425,363 1424,363 1423,364 1422,363 1421,364 1420,364 1419,364 1418,365 1417,364 1416,364 1415,364 1414,363 1413,363 1412,363 1411,362 1410,361 1409,360 1408,359 1407,359 1406,359 1405,359 1404,359 1403,358 1402,358 1401,359 1400,360 1399,361 1398,362 1397,362 1396,362 1395,362 1394,362 1393,362 1392,362 1391,363 1390,362 1389,363 1388,364 1387,364 1386,364 1385,363 1384,364 1383,363 1382,363 1381,362 1380,361 1379,360 1378,359 1377,358 1376,357 1375,356 1374,357 1373,357 1372,357 1371,356 1370,357 1369,358 1368,359 1367,360 1366,361 1365,362 1364,363 1363,364 1362,364 1361,363 1360,363 1359,363 1358,364 1357,365 1356,365 1355,365 1354,365 1353,364 1352,365 1351,365 1350,365 1349,366 1348,365 1347,365 1346,364 1345,365 1344,364 1343,363 1342,362 1341,361 1340,360 1339,361 1338,362 1337,363 1336,364 1335,363 1334,364 1333,364 1332,364 1331,364 1330,364 1329,364 1328,364 1327,364 1326,364 1325,364 1324,365 1323,365 1322,365 1321,364 1320,364 1319,363 1318,363 1317,363 1316,362 1315,361 1314,360 1313,359 1312,358 1311,357 1310,356 1309,355 1308,355 1307,355 1306,355 1305,355 1304,354 1303,353 1302,354 1301,354 1300,354 1299,354 1298,354 1297,353 1296,353 1295,354 1294,354 1293,353 1292,353 1291,353 1290,353 1289,353 1288,354 1287,355 1286,356 1285,357 1284,356 1283,355 1282,355 1281,354 1280,355 1279,355 1278,354 1277,354 1276,354 1275,353 1274,353 1273,354 1272,353 1271,354 1270,355 1269,355 1268,354 1267,355 1266,354 1265,355 1264,355 1263,355 1262,355 1261,356 1260,356 1259,355 1258,356 1257,357 1256,358 1255,359 1254,360 1253,361 1252,362 1251,363 1250,364 1249,364 1248,364 1247,364 1246,364 1245,364 1244,364 1243,364 1242,364 1241,364 1240,364 1239,363 1238,363 1237,363 1236,362 1235,362 1234,363 1233,363 1232,363 1231,364 1230,363 1229,364 1228,364 1227,363 1226,362 1225,362 1224,361 1223,360 1222,359 1221,358 1220,357 1219,357 1218,356 1217,355 1216,356 1215,356 1214,357 1213,357 1212,357 1211,356 1210,355 1209,354 1208,353 1207,353 1206,353 1205,354 1204,353 1203,353 1202,353 1201,352 1200,352 1199,353 1198,353 1197,353 1196,354 1195,355 1194,356 1193,357 1192,356 1191,356 1190,357 1189,357 1188,357 1187,357 1186,357 1185,357 1184,356 1183,355 1182,354 1181,354 1180,355 1179,355 1178,354 1177,355 1176,355 1175,354 1174,354 1173,354 1172,353 1171,353 1170,353 1169,353 1168,354 1167,355 1166,356 1165,357 1164,358 1163,358 1162,359 1161,358 1160,357 1159,356 1158,355 1157,354 1156,354 1155,354 1154,355 1153,354 1152,355 1151,355 1150,355 1149,356 1148,356 1147,355 1146,354 1145,353 1144,352 1143,351 1142,351 1141,351 1140,352 1139,351 1138,350 1137,351 1136,352 1135,353 1134,353 1133,353 1132,353 1131,352 1130,351 1129,350 1128,349 1127,348 1126,347 1125,346 1124,345 1123,345 1122,346 1121,347 1120,347 1119,346 1118,347 1117,348 1116,349 1115,350 1114,351 1113,352 1112,353 1111,354 1110,355 1109,356 1108,357 1107,358 1106,359 1105,360 1104,361 1103,362 1102,363 1101,364 1100,363 1099,362 1098,362 1097,363 1096,364 1095,363 1094,364 1093,364 1092,364 1091,365 1090,366 1089,366 1088,366 1087,366 1086,367 1085,368 1084,369 1083,370 1082,371 1081,372 1080,372 1079,372 1078,372 1077,371 1076,371 1075,372 1074,373 1073,374 1072,374 1071,374 1070,374 1069,374 1068,374 1067,374 1066,374 1065,374 1064,374 1063,374 1062,374 1061,374 1060,373 1059,372 1058,371 1057,370 1056,369 1055,369 1054,370 1053,369 1052,370 1051,371 1050,370 1049,369 1048,370 1047,371 1046,372 1045,371 1044,371 1043,371 1042,370 1041,370 1040,371 1039,370 1038,369 1037,368 1036,367 1035,366 1034,366 1033,367 1032,367 1031,366 1030,367 1029,366 1028,366 1027,367 1026,368 1025,368 1024,367 1023,366 1022,365 1021,364 1020,363 1019,363 1018,364 1017,364 1016,363 1015,363 1014,362 1013,361 1012,360 1011,359 1010,358 1009,357 1008,356 1007,355 1006,354 1005,353 1004,353 1003,352 1002,351 1001,350 1000,349 999,348 998,347 997,346 996,345 995,345 994,345 993,345 992,345 991,345 990,345 989,345 988,345 987,345 986,344 985,343 984,342 983,341 982,340 981,339 980,338 979,337 978,336 977,335 976,334 975,333 974,332 973,332 972,331 971,330 970,329 969,328 968,327 967,326 966,325 965,324 964,323 963,322 962,322 961,322 960,322 959,322 958,322 957,322 956,323 955,324 954,324 953,323 952,322 951,322 950,322 949,322 948,322 947,322 946,322 945,322 944,322 943,322 942,322 941,322 940,322 939,322 938,322 937,322 936,322 935,322 934,322 933,322 932,322 931,322 930,322 929,322 928,322 927,323 926,323 925,324 924,325 923,324 922,324 921,323 920,322 919,322 918,322 917,322 916,322 915,322 914,322 913,322 912,322 911,322 910,322 909,322 908,322 907,322 906,322 905,322 904,322 903,322 902,322 901,322 900,322 899,322 898,322 897,322 896,322 895,322 894,322 893,322 892,323 891,323 890,323 889,323 888,323 887,323 886,323 885,323 884,323 883,323 882,323 881,323 880,323 879,324 878,323 877,323 876,323 875,323 874,323 873,323 872,323 871,324 870,323 869,323 868,323 867,323 866,323 865,323 864,323 863,323 862,323 861,323 860,323 859,324 858,325 857,325 856,325 855,324 854,323 853,323 852,323 851,323 850,323 849,323 848,324 847,325 846,326 845,326 844,326 843,326 842,327 841,328 840,327 839,326 838,327 837,326 836,325 835,324 834,323 833,323 832,323 831,323 830,323 829,323 828,323 827,323 826,323 825,323 824,324 823,325 822,326 821,327 820,328 819,329 818,330 817,331 816,332 815,333 814,334 813,334 812,334 811,334 810,334 809,334 808,334 807,334 806,334 805,334 804,334 803,335 802,335 801,335 800,334 799,333 798,332 797,331 796,330 795,329 794,328 793,327 792,326 791,325 790,324"
points = [tuple(map(int, pt.split(','))) for pt in points.split()]

In [46]:
points

[(790, 313),
 (791, 312),
 (792, 311),
 (793, 310),
 (794, 311),
 (795, 312),
 (796, 313),
 (797, 314),
 (798, 315),
 (799, 314),
 (800, 313),
 (801, 313),
 (802, 312),
 (803, 312),
 (804, 311),
 (805, 310),
 (806, 309),
 (807, 310),
 (808, 311),
 (809, 312),
 (810, 313),
 (811, 314),
 (812, 314),
 (813, 313),
 (814, 313),
 (815, 312),
 (816, 311),
 (817, 311),
 (818, 311),
 (819, 312),
 (820, 313),
 (821, 313),
 (822, 314),
 (823, 313),
 (824, 312),
 (825, 313),
 (826, 313),
 (827, 314),
 (828, 315),
 (829, 315),
 (830, 315),
 (831, 315),
 (832, 315),
 (833, 316),
 (834, 316),
 (835, 315),
 (836, 315),
 (837, 315),
 (838, 315),
 (839, 315),
 (840, 314),
 (841, 314),
 (842, 314),
 (843, 314),
 (844, 315),
 (845, 315),
 (846, 315),
 (847, 314),
 (848, 314),
 (849, 314),
 (850, 315),
 (851, 315),
 (852, 315),
 (853, 316),
 (854, 317),
 (855, 317),
 (856, 316),
 (857, 315),
 (858, 315),
 (859, 314),
 (860, 314),
 (861, 314),
 (862, 315),
 (863, 315),
 (864, 315),
 (865, 316),
 (866, 316),